# FinQA preprocessing validation and canonical-answer policy

This notebook validates existing preprocessing against real train annotations and documented malformed inputs. It preserves the pinned raw files and split assignments. The answer-policy decision is based on train annotations only; the official test split is not inspected for baseline development or tolerance selection.

In [1]:
from __future__ import annotations

import json
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd

from finevidence.data import load_finqa_split, normalize_financial_value, parse_finqa_program

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
RAW_DIR = PROJECT_ROOT / "data" / "raw"
REPORT_PATH = PROJECT_ROOT / "data" / "processed" / "m2_preprocessing_diagnostics.json"
train_records = load_finqa_split(RAW_DIR / "train.json")
diagnostic_report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
print("Train records used for all policy decisions:", len(train_records))
print("Previously recorded split-level parser coverage:", diagnostic_report["parser_coverage"])

Train records used for all policy decisions: 6251
Previously recorded split-level parser coverage: {'dev': {'programs_parsed': 883, 'records': 883}, 'test': {'programs_parsed': 1147, 'records': 1147}, 'train': {'programs_parsed': 6251, 'records': 6251}}


## 1. Program-parser structure

Parsing is not execution. The structural check below verifies call order, two-argument boundaries, table-label handling, and that an intermediate reference points only to an earlier parsed step. The latter is intentionally separate from the current parser's token-level acceptance.

In [2]:
def reference_order_errors(result):
    errors = []
    for step_index, step in enumerate(result.steps):
        for argument in step.arguments:
            if argument.startswith("#") and int(argument[1:]) >= step_index:
                errors.append((step_index, argument))
    return errors


program_ids = [
    "AWK/2014/page_122.pdf-3",
    "HOLX/2005/page_27.pdf-2",
    "ALLE/2016/page_29.pdf-1",
    "RSG/2008/page_114.pdf-1",
]
real_program_rows = []
for record_id in program_ids:
    record = next(item for item in train_records if item["id"] == record_id)
    result = parse_finqa_program(record["qa"]["program"])
    real_program_rows.append(
        {
            "id": record_id,
            "program": record["qa"]["program"],
            "parse_valid": result.is_valid,
            "operations_in_order": [step.operation for step in result.steps],
            "arguments": [step.arguments for step in result.steps],
            "argument_kinds": [step.argument_kinds for step in result.steps],
            "reference_order_errors": reference_order_errors(result),
        }
    )
display(pd.DataFrame(real_program_rows))
assert all(row["parse_valid"] and not row["reference_order_errors"] for row in real_program_rows)

,id,program,parse_valid,operations_in_order,arguments,argument_kinds,reference_order_errors
0,AWK/2014/page_122.pdf-3,"multiply(3176, const_m1), divide(#0, 10379)",True,"[multiply, divide]","[(3176, const_m1), (#0, 10379)]","[(numeric_literal, named_constant), (intermedi...",[]
1,HOLX/2005/page_27.pdf-2,"table_sum(17695228, 963202), add(#0, 155213), ...",True,"[table_sum, add, divide]","[(17695228, 963202), (#0, 155213), (17695228, ...","[(numeric_literal, numeric_literal), (intermed...",[]
2,ALLE/2016/page_29.pdf-1,"table_average(2016, none)",True,[table_average],"[(2016, none)]","[(table_label, none)]",[]
3,RSG/2008/page_114.pdf-1,"divide(285.7, .76)",True,[divide],"[(285.7, .76)]","[(numeric_literal, numeric_literal)]",[]


In [3]:
structural_rows = []
for record in train_records:
    result = parse_finqa_program(record["qa"]["program"])
    structural_rows.append(
        {
            "parsed": result.is_valid,
            "reference_order_valid": result.is_valid and not reference_order_errors(result),
            "steps": len(result.steps),
        }
    )
structural_frame = pd.DataFrame(structural_rows)
display(
    structural_frame[["parsed", "reference_order_valid"]]
    .value_counts()
    .rename("programs")
    .to_frame()
)
display(structural_frame["steps"].value_counts().sort_index().rename("programs").to_frame())

malformed_programs = [
    "add(1)",
    "add(1, 2",
    "mean(1, 2)",
    "table_sum(revenue, total)",
    "add(1, #1)",
]
malformed_rows = []
for program in malformed_programs:
    result = parse_finqa_program(program)
    malformed_rows.append(
        {
            "program": program,
            "parser_accepts_syntax": result.is_valid,
            "reference_order_errors": reference_order_errors(result),
            "parse_error": result.error,
        }
    )
display(pd.DataFrame(malformed_rows))

assert structural_frame["parsed"].all()
assert structural_frame["reference_order_valid"].all()
assert any(row["reference_order_errors"] for row in malformed_rows)

,,programs
parsed,reference_order_valid,
True,True,6251


,programs
steps,
1,3717
2,2013
3,331
4,94
5,90
6,6


,program,parser_accepts_syntax,reference_order_errors,parse_error
0,add(1),False,[],expected two non-empty arguments in: 'add(1)'
1,"add(1, 2",False,[],unbalanced parentheses
2,"mean(1, 2)",False,[],unsupported operation: 'mean'
3,"table_sum(revenue, total)",False,[],unsupported table arguments in: 'table_sum(rev...
4,"add(1, #1)",True,"[(0, #1)]",NaN


## 2. Conservative normalization audit

Only train table cells are inspected here. Unsupported strings are not errors and do not trigger new rules automatically; a new rule would require a stable, unambiguous observed form and a matching unit test.

In [4]:
unsupported_rows = []
for record in train_records:
    for row_index, row in enumerate(record["table"]):
        for column_index, raw in enumerate(row):
            candidate = normalize_financial_value(str(raw))
            if candidate.status == "unsupported" and re.search(r"[0-9$%]", str(raw)):
                unsupported_rows.append(
                    {
                        "id": record["id"],
                        "cell": f"table[{row_index}][{column_index}]",
                        "raw": raw,
                    }
                )

unsupported_frame = pd.DataFrame(unsupported_rows)
display(unsupported_frame.head(12))
print("Numeric-looking unsupported train cells:", len(unsupported_frame))
print(
    "Decision: retain these raw forms as unsupported; no normalization rule is added in this notebook."
)

,id,cell,raw
0,ADI/2009/page_49.pdf-1,table[1][2],$ -23158 ( 23158 )
1,ADI/2009/page_49.pdf-1,table[2][0],fair value of forward exchange contracts after...
2,ADI/2009/page_49.pdf-1,table[2][2],$ -9457 ( 9457 )
3,ADI/2009/page_49.pdf-1,table[3][0],fair value of forward exchange contracts after...
4,ADI/2009/page_49.pdf-1,table[3][1],$ -6781 ( 6781 )
5,ADI/2009/page_49.pdf-1,table[3][2],$ -38294 ( 38294 )
6,ABMD/2012/page_75.pdf-1,table[3][1],-134 ( 134 )
7,ABMD/2012/page_75.pdf-1,table[4][1],-9 ( 9 )
8,AAL/2018/page_13.pdf-2,table[1][4],23.6% ( 23.6 % )
9,AAL/2018/page_13.pdf-2,table[2][4],19.6% ( 19.6 % )


Numeric-looking unsupported train cells: 28046
Decision: retain these raw forms as unsupported; no normalization rule is added in this notebook.


## 3. Train-only canonical-answer policy

The following inspection classifies raw `answer` and `exe_ans` annotations on train only. It does not select a model, inspect test labels, or fit an answer normalizer. Numeric comparison is allowed only for isolated numeric answer strings; all other non-empty answers use conservative case- and whitespace-normalized string comparison.

In [5]:
NUMERIC_ANSWER = re.compile(
    r"^\s*(?P<sign>[+-])?(?P<number>(?:\d+(?:,\d{3})*(?:\.\d+)?|\.\d+))(?P<percent>%?)\s*$"
)


def canonical_text(value):
    return " ".join(value.strip().casefold().split())


def parse_isolated_numeric_answer(value):
    match = NUMERIC_ANSWER.fullmatch(value)
    if match is None:
        return None
    try:
        magnitude = Decimal(match["number"].replace(",", ""))
    except InvalidOperation:
        return None
    if match["sign"] == "-":
        magnitude = -magnitude
    return magnitude, bool(match["percent"])


answer_rows = []
for record in train_records:
    answer = record["qa"]["answer"]
    numeric = parse_isolated_numeric_answer(answer)
    normalized = canonical_text(answer)
    answer_type = (
        "empty_display_answer"
        if not normalized
        else "yes_no"
        if normalized in {"yes", "no"}
        else "numeric_percent"
        if numeric and numeric[1]
        else "numeric"
        if numeric
        else "other_text"
    )
    answer_rows.append(
        {
            "id": record["id"],
            "answer": answer,
            "exe_ans": record["qa"]["exe_ans"],
            "answer_type": answer_type,
            "numeric_value": numeric[0] if numeric else None,
            "percent_marker": numeric[1] if numeric else False,
        }
    )
answer_frame = pd.DataFrame(answer_rows)
display(answer_frame["answer_type"].value_counts().rename("train_records").to_frame())
display(
    answer_frame.groupby("answer_type").head(3)[
        ["id", "answer", "exe_ans", "numeric_value", "percent_marker"]
    ]
)

,train_records
answer_type,
numeric_percent,3471
numeric,2464
other_text,148
yes_no,120
empty_display_answer,48


,id,answer,exe_ans,numeric_value,percent_marker
0,ADI/2009/page_49.pdf-1,380,3.8,380,False
1,ABMD/2012/page_75.pdf-1,,yes,None,False
2,AAL/2018/page_13.pdf-2,41932,41932.20339,41932,False
3,INTC/2013/page_71.pdf-4,53%,0.53232,53,True
4,ETR/2008/page_313.pdf-3,-3.2%,-0.03219,-3.2,True
5,C/2010/page_272.pdf-1,56.25%,0.97656,56.25,True
6,AMT/2012/page_121.pdf-1,7.4,7.385,7.4,False
11,PNC/2012/page_110.pdf-3,,22929.0,None,False
28,TSCO/2017/page_68.pdf-3,$ 13 million,13.0,None,False
47,JPM/2018/page_73.pdf-5,yes,yes,None,False


In [6]:
numeric_pairs = []
for row in answer_rows:
    if row["numeric_value"] is None or not isinstance(row["exe_ans"], (int, float)):
        continue
    displayed = row["numeric_value"]
    executable = Decimal(str(row["exe_ans"]))
    numeric_pairs.append(
        {
            "id": row["id"],
            "answer": row["answer"],
            "exe_ans": row["exe_ans"],
            "percent_marker": row["percent_marker"],
            "direct_match": displayed == executable,
            "percent_scaled_match": row["percent_marker"]
            and displayed / Decimal("100") == executable,
        }
    )
numeric_pair_frame = pd.DataFrame(numeric_pairs)
if numeric_pair_frame.empty:
    raise RuntimeError("No train numeric answer/exe_ans pairs were observed.")
display(
    numeric_pair_frame[["direct_match", "percent_scaled_match"]]
    .value_counts()
    .rename("train_pairs")
    .to_frame()
)
display(
    numeric_pair_frame[
        ~(numeric_pair_frame["direct_match"] | numeric_pair_frame["percent_scaled_match"])
    ].head(8)
)

ABSOLUTE_TOLERANCE = Decimal("0.0001")
RELATIVE_TOLERANCE = Decimal("0.0001")


def numerically_equivalent(prediction, reference):
    difference = abs(prediction - reference)
    scale = max(abs(reference), Decimal("1"))
    return difference <= max(ABSOLUTE_TOLERANCE, RELATIVE_TOLERANCE * scale)


print("Train annotations show that answer and exe_ans are not uniformly directly comparable.")
print("Pre-registered display-answer tolerance: max(1e-4 absolute, 1e-4 relative).")
print("It applies only to a model numeric prediction versus a numeric display answer.")

,,train_pairs
direct_match,percent_scaled_match,
False,False,4060
True,False,1556
False,True,312
True,True,7


,id,answer,exe_ans,percent_marker,direct_match,percent_scaled_match
0,ADI/2009/page_49.pdf-1,380,3.80000,False,False,False
1,AAL/2018/page_13.pdf-2,41932,41932.20339,False,False,False
2,INTC/2013/page_71.pdf-4,53%,0.53232,True,False,False
3,ETR/2008/page_313.pdf-3,-3.2%,-0.03219,True,False,False
4,C/2010/page_272.pdf-1,56.25%,0.97656,True,False,False
5,AMT/2012/page_121.pdf-1,7.4,7.38500,False,False,False
6,GIS/2019/page_45.pdf-1,63.6%,0.63634,True,False,False
7,IPG/2009/page_89.pdf-3,96.55%,96.55172,True,False,False


Train annotations show that answer and exe_ans are not uniformly directly comparable.
Pre-registered display-answer tolerance: max(1e-4 absolute, 1e-4 relative).
It applies only to a model numeric prediction versus a numeric display answer.


## Decision: evaluation denominator and comparison rules

1. **Display-answer evaluation denominator:** all train/dev examples with non-empty `answer`; report the count of empty display answers separately. Do not silently remove them.
2. **Numeric display answers:** parse only isolated signed decimal/integer strings with an optional `%`. Compare a model numeric prediction with the numeric display answer using pre-registered `max(1e-4 absolute, 1e-4 relative)` tolerance, preserving the percent marker as part of the answer type. Do not compare a percent answer to an unmarked decimal without an explicitly documented task-level conversion. The train audit shows that `answer` and `exe_ans` are not uniformly directly comparable, so they are not used to calibrate this tolerance.
3. **Non-numeric display answers:** compare case-folded, whitespace-collapsed strings exactly. This covers observed yes/no answers and other text without forcing them through a numeric parser.
4. **Empty display answers:** keep the examples in the dataset and report them as `display-answer unavailable`; exclude them only from the display-answer metric denominator. A later execution/program metric may use `exe_ans`, but its denominator and normalization must be reported separately.
5. **Formatting and rounding:** commas and leading-zero formatting are accepted only within isolated numeric answers; currency, unit, date, and mixed strings stay text unless a later, tested policy expands coverage.

This policy was selected from train annotations only. Dev may be used later for model selection under this fixed policy; test remains reserved for final reporting.